# Customer Segmentation of Travel Reviewers

Unsupervised segmentation of 5,455 Google travel reviewers across 24 attraction
categories, using k-means with PCA and UMAP, to define target segments for
tailored European travel packages.

Georgetown MSBA · Clustering & Segmentation · Team project.

## Step 0: Import Packages and Data
---
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap

from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score, silhouette_samples, adjusted_rand_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import cross_val_score

from matplotlib.colors import ListedColormap
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
# Plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Single seed used throughout for reproducibility.
RANDOM_SEED = 315
np.random.seed(RANDOM_SEED)

In [ ]:
# Travel_Review.xlsx is committed alongside this notebook.
df = pd.read_excel('Travel_Review.xlsx')
N_ORIGINAL = len(df)
print(f'Loaded {N_ORIGINAL} reviewers')

## Step 1: Pre-Processing
---
---

In [ ]:
print(f"Shape: {df.shape}")
print(df.info())
print(df.head())

In [ ]:
df['Gardens'] = df['Gardens'].fillna(0)

In [ ]:
df.isna().sum()

In [ ]:
users_cols = ["UserID"]
reviews_cols = [
    "Churches", "Resorts", "Beaches", "Parks", "Theatres", "Museums",
    "Malls", "Zoo", "Restaurants", "Pubs_Bars", "LocalServices",
    "Burger_PizzaShops", "Hotels_OtherLodgings", "JuiceBars",
    "ArtGalleries", "DanceClubs", "Swimming Pools", "Gyms", "Bakeries",
    "BeautySpas", "Cafes", "ViewPoints", "Monuments", "Gardens"
]
df_users = df[users_cols].copy()
df_reviews = df[reviews_cols].copy()

print("Clustering features:", reviews_cols)
print("Context variables (not used in clustering):", users_cols)

In [ ]:
# --- Remove low-engagement users (too many zero ratings) ---
# A zero means the user never reviewed that category, so users who are mostly
# zeros carry little signal about preference and distort distance calculations.

zero_frac = (df[reviews_cols] == 0).mean(axis=1)
MAX_ZERO_FRAC = 0.25
keep_mask = zero_frac <= MAX_ZERO_FRAC

dropped_zero_frac = zero_frac[~keep_mask].mean()

df = df.loc[keep_mask].reset_index(drop=True)

# Rebuild the analysis frames from the FILTERED data. Everything downstream
# (PCA, Yeo-Johnson, scaling, k-means) reads from df_reviews, so this line is
# what makes the exclusion actually take effect.
df_reviews = df[reviews_cols].copy()
df_users = df[users_cols].copy()
X = df_reviews.copy()

print(f'Original users: {N_ORIGINAL}')
print(f'Kept users:     {len(df)}')
print(f'Dropped users:  {N_ORIGINAL - len(df)}')
print(f'Mean zero fraction (kept):    {(X == 0).mean(axis=1).mean():.3f}')
print(f'Mean zero fraction (dropped): {dropped_zero_frac:.3f}')

## Step 3: Exploratory Data Analysis
---
---

In [ ]:
n_pcs = 8

In [ ]:
df_reviews.describe().round(1)

fig, axes = plt.subplots(5, 5, figsize=(18, 14))
for ax, col in zip(axes.ravel(), reviews_cols):
    ax.hist(df_reviews[col], bins=40, edgecolor="white", color="steelblue")
    ax.set_title(col)
    ax.set_xlabel("Rating")
fig.suptitle("Distributions of Review Variables (Full Dataset)", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------
# t-SNE & UMAP (on full data)
# ---------------------------------------------------

# 1) Ensure numeric
X = X.apply(pd.to_numeric, errors="coerce")

# 2) Treat missing as 0
X = X.fillna(0)

# 3) Drop constant columns
stds = X.std(axis=0)
constant_cols = stds[stds == 0].index.tolist()
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

# 4) Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5) Safety check
print("Any NaNs in X_scaled?", np.isnan(X_scaled).any())
print("Shape:", X_scaled.shape)

### t-SNE

In [ ]:
# t-SNE
TSNE_PERPLEXITY = 30
TSNE_LEARNING_RATE = "auto"
TSNE_N_ITER = 1500

tsne = TSNE(
    n_components=2,
    perplexity=TSNE_PERPLEXITY,
    learning_rate=TSNE_LEARNING_RATE,
    max_iter=TSNE_N_ITER,
    init="pca",
    random_state=RANDOM_SEED
)

tsne_2d = tsne.fit_transform(X_scaled)
tsne_2d.shape

### UMAP

In [ ]:
# UMAP
UMAP_N_NEIGHBORS = 25
UMAP_MIN_DIST = 0.10
UMAP_METRIC = "euclidean"

umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_SEED
)

umap_2d = umap_model.fit_transform(X_scaled)
umap_2d.shape

In [ ]:
# Cluster on embedding
K = 8
kmeans_tsne = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init="auto").fit(tsne_2d)
kmeans_umap = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init="auto").fit(umap_2d)

labels_tsne = kmeans_tsne.labels_
labels_umap = kmeans_umap.labels_

(labels_tsne[:10], labels_umap[:10])

### Cluster Analysis

In [ ]:
# Plotting
def scatter_plot(emb, labels=None, title="Embedding"):
    plt.figure(figsize=(8, 6))

    if labels is None:
        plt.scatter(emb[:, 0], emb[:, 1], s=8, color="gray")
    else:
        colors = ["#4E79A7", "#B07AA1", "#59A14F", "#F4A261", "#ffb068", "#8ff6ff", "#ff00ff", "#BEF527"]   # NEED TO ADD MORE COLORS
        cmap = ListedColormap(colors)
        plt.scatter(emb[:, 0], emb[:, 1], s=8, c=labels, cmap=cmap)

    plt.title(title)
    plt.xlabel("Dim 1")
    plt.ylabel("Dim 2")
    plt.tight_layout()
    plt.show()

scatter_plot(tsne_2d, labels_tsne, f"t-SNE (perplexity={TSNE_PERPLEXITY}) with KMeans K={K}")
scatter_plot(umap_2d, labels_umap, f"UMAP (n_neighbors={UMAP_N_NEIGHBORS}, min_dist={UMAP_MIN_DIST}) with KMeans K={K}")

In [ ]:
# Average ratings per cluster (using UMAP clusters)
profile = df.copy()
profile["cluster"] = labels_umap

cluster_means = profile.groupby("cluster")[reviews_cols].mean()
cluster_sizes = profile["cluster"].value_counts().sort_index()

display(cluster_sizes)
display(cluster_means)

### Comparison

In [ ]:
# t-SNE cluster profiling
cluster_profile_tsne = df.copy()
cluster_profile_tsne["cluster"] = labels_tsne
means_tsne = cluster_profile_tsne.groupby("cluster")[reviews_cols].mean()
means_tsne

In [ ]:
# UMAP cluster profiling (again)
cluster_profile_umap = df.copy()
cluster_profile_umap["cluster"] = labels_umap
cluster_means_umap = cluster_profile_umap.groupby("cluster")[reviews_cols].mean()
cluster_sizes_umap = cluster_profile_umap["cluster"].value_counts()
display(cluster_sizes_umap)
display(cluster_means_umap)

### PCA

In [ ]:
# ---------------------------------------------------
# PCA (on full data, no hold-out)
# ---------------------------------------------------

meta_groups = {
    "Cultural_Heritage": ["Churches", "Museums", "Theatres", "ArtGalleries", "Monuments", "ViewPoints"],
    "Nature_Outdoor": ["Parks", "Beaches", "Gardens", "Zoo"],
    "Shopping": ["Malls", "Bakeries", "LocalServices"],
    "Food_Beverage": ["Restaurants", "Pubs_Bars", "Burger_PizzaShops", "Cafes", "JuiceBars"],
    "Accommodation": ["Hotels_OtherLodgings", "Resorts"],
    "Recreation_Wellness": ["DanceClubs", "Swimming Pools", "Gyms", "BeautySpas"],
}

meta_df = pd.DataFrame(index=df.index)
for name, cols in meta_groups.items():
    meta_df[name] = df[cols].mean(axis=1)

meta_df.describe()

In [ ]:
pca_full = PCA()
pca_full.fit(df_reviews)

explained = pca_full.explained_variance_ratio_
cum_explained = np.cumsum(explained)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, len(explained) + 1), explained, marker='o')
ax1.set_xlabel('PC')
ax1.set_ylabel('Variance ratio')
ax1.set_title('Scree plot')

ax2.plot(range(1, len(cum_explained) + 1), cum_explained, marker='o')
for level, color in [(0.8, 'r'), (0.85, 'g'), (0.9, 'b')]:
    ax2.axhline(level, color=color, ls='--', linewidth=1)
ax2.set_xlabel('PC')
ax2.set_ylabel('Cumulative variance')
ax2.set_title('Cumulative variance explained')

fig.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=n_pcs)
X_pca = pca.fit_transform(df_reviews)

loadings = pd.DataFrame(
    pca.components_.T,
    index=reviews_cols,
    columns=[f"PC{i+1}" for i in range(n_pcs)]
)

for i in range(min(3, n_pcs)):
    pc = f"PC{i+1}"
    print(f"\nTop 10 features for {pc}:")
    print(loadings[pc].sort_values(key=np.abs, ascending=False).head(10))

# PCA on meta features
pca_meta = PCA()
X_meta_pca = pca_meta.fit_transform(meta_df)

meta_expl = pca_meta.explained_variance_ratio_
meta_cum = np.cumsum(meta_expl)

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(meta_expl) + 1), meta_expl, marker='o')
plt.plot(range(1, len(meta_cum) + 1), meta_cum, marker='o')
plt.axhline(0.85, color='r', ls='--')
plt.xlabel("PC")
plt.ylabel("Variance / Cumulative")
plt.title("PCA on meta-features")
plt.grid(True)
plt.show()

meta_loadings = pd.DataFrame(
    pca_meta.components_.T,
    index=meta_df.columns,
    columns=[f"PC{i+1}" for i in range(len(meta_expl))]
)
meta_loadings

In [ ]:
corr = df_reviews.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Correlation Matrix -- Review Variables (Full Dataset)")
plt.tight_layout()
plt.show()

## Step 4: Feature Engineering
---
---

In [ ]:
# Step 4: Feature Engineering on full data
pt = PowerTransformer(method='yeo-johnson', standardize=False)

X_yj = df_reviews.copy()
X_yj[reviews_cols] = pt.fit_transform(df_reviews[reviews_cols])

fig, axes = plt.subplots(5, 5, figsize=(18, 14))
for ax, col in zip(axes.ravel(), reviews_cols):
    ax.hist(X_yj[col], bins=40, edgecolor="white", color="teal")
    ax.set_title(f"Yeo-Johnson ({col})")
fig.suptitle("Distributions After Yeo-Johnson Transform (Full Dataset)", fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled_full = scaler.fit_transform(X_yj)

X_scaled_full = pd.DataFrame(X_scaled_full, columns=reviews_cols, index=df_reviews.index)

print("Scaled full set -- means and std devs:")
print(X_scaled_full.describe().loc[["mean", "std"]].round(4))

In [ ]:
meta_scaler = StandardScaler()
X_meta_scaled = meta_scaler.fit_transform(meta_df.values)

## Step 5: Clustering
---
---

### k-Means

In [ ]:
# Step 5: K-Means model selection on full data
K_range = range(2, 21)
wcss_scores = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_SEED)
    labels = km.fit_predict(X_scaled_full)
    wcss_scores.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled_full, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, wcss_scores, "o-", color="steelblue", linewidth=2)
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("WCSS (Inertia)")
ax1.set_title("Elbow Plot")
ax1.set_xticks(list(K_range))

ax2.plot(K_range, sil_scores, "o-", color="teal", linewidth=2)
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Average Silhouette Score")
ax2.set_title("Silhouette Scores")
ax2.set_xticks(list(K_range))

fig.suptitle("K-Means Diagnostics Across k (Full Data)", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Choose candidate k
k_7 = 7
print(f"Selected k = {k_7}")
print(f"  WCSS:       {wcss_scores[k_7 - 2]:.1f}")
print(f"  Silhouette: {sil_scores[k_7 - 2]:.4f}")

km_7 = KMeans(n_clusters=k_7, n_init=20, random_state=RANDOM_SEED)
labels_7 = km_7.fit_predict(X_scaled_full)
print("Cluster sizes (k=7, full data):")
print(pd.Series(labels_7).value_counts().sort_index())

In [ ]:
k_9 = 9
print(f"Selected k = {k_9}")
print(f"  WCSS:       {wcss_scores[k_9 - 2]:.1f}")
print(f"  Silhouette: {sil_scores[k_9 - 2]:.4f}")

In [ ]:
k_8 = 8
print(f"Selected k = {k_8}")
print(f"  WCSS:       {wcss_scores[k_8 - 2]:.1f}")
print(f"  Silhouette: {sil_scores[k_8 - 2]:.4f}")

km_8 = KMeans(n_clusters=k_8, n_init=20, random_state=RANDOM_SEED)
labels_8 = km_8.fit_predict(X_scaled_full)
print("Cluster sizes (k=8, full data):")
print(pd.Series(labels_8).value_counts().sort_index())

k_chosen = 8
km_final = KMeans(n_clusters=k_chosen, n_init=20, random_state=RANDOM_SEED)
cluster_labels = km_final.fit_predict(X_scaled_full)

print("Cluster sizes (chosen k, full data):")
print(pd.Series(cluster_labels).value_counts().sort_index())

In [ ]:
# Silhouette analysis on full data
sil_vals = silhouette_samples(X_scaled_full, cluster_labels)
avg_sil = sil_vals.mean()

fig, ax = plt.subplots(figsize=(10, 7))
y_lower = 10

for i in range(k_chosen):
    cluster_sil = np.sort(sil_vals[cluster_labels == i])
    size_i = cluster_sil.shape[0]
    y_upper = y_lower + size_i

    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     alpha=0.7, label=f"Cluster {i} (n={size_i})")
    ax.text(-0.05, y_lower + 0.5 * size_i, str(i), fontweight="bold")
    y_lower = y_upper + 10

ax.axvline(avg_sil, color="red", linestyle="--", label=f"Mean = {avg_sil:.3f}")
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Observations (sorted within cluster)")
ax.set_title("Silhouette Plot by Cluster (Full Data)")
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
n_neg = (sil_vals < 0).sum()
print(f"Points with negative silhouette: {n_neg} / {len(sil_vals)} ({100*n_neg/len(sil_vals):.1f}%)")

In [ ]:
# Stability via resampling on full data
B = 30
ari_scores = []

for b in range(B):
    idx = np.random.choice(X_scaled_full.index, size=int(0.8 * len(X_scaled_full)),
                           replace=False)
    X_sub = X_scaled_full.loc[idx]

    km_sub = KMeans(n_clusters=k_chosen, n_init=20, random_state=b)
    sub_labels = km_sub.fit_predict(X_sub)

    ref_labels = pd.Series(cluster_labels, index=X_scaled_full.index).loc[idx].values
    ari = adjusted_rand_score(ref_labels, sub_labels)
    ari_scores.append(ari)

print(f"Stability (ARI) over {B} runs:")
print(f"  Mean:   {np.mean(ari_scores):.3f}")
print(f"  Std:    {np.std(ari_scores):.3f}")
print(f"  Min:    {np.min(ari_scores):.3f}")
print(f"  Max:    {np.max(ari_scores):.3f}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(ari_scores, bins=15, edgecolor="white", color="steelblue")
plt.axvline(np.mean(ari_scores), color="red", linestyle="--",
            label=f"Mean ARI = {np.mean(ari_scores):.3f}")
plt.xlabel("Adjusted Rand Index")
plt.ylabel("Count")
plt.title("Clustering Stability: ARI Across Resampled Datasets (Full Data)")
plt.legend()
plt.tight_layout()
plt.show()

### Hierarchical

In [ ]:
# ---------------------------------------------------
# Hierarchical Clustering (Full Data)
# ---------------------------------------------------

Z = linkage(X_scaled_full, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90,
           leaf_font_size=10, color_threshold=0)
plt.title("Hierarchical Clustering Dendrogram (Ward's Linkage)")
plt.xlabel("Cluster Size")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()


In [ ]:
hc = AgglomerativeClustering(n_clusters=k_chosen, linkage="ward")
hc_labels = hc.fit_predict(X_scaled_full)

ari_hc = adjusted_rand_score(cluster_labels, hc_labels)
print(f"ARI between K-Means and Hierarchical (Ward's): {ari_hc:.3f}")

print("\nCrosstab of K-Means vs. Hierarchical assignments:")
print(pd.crosstab(
    pd.Series(cluster_labels, name="K-Means"),
    pd.Series(hc_labels, name="Hierarchical")
))

## Step 6: Interpretation and Evaluation
---
---

In [ ]:
# ---------------------------------------------------
# Cluster Interpretation and Evaluation (Full Data)
# ---------------------------------------------------

# Means of scaled features by cluster
profile_scaled = X_scaled_full.copy()
profile_scaled["Cluster"] = cluster_labels

cluster_means_scaled = profile_scaled.groupby("Cluster")[reviews_cols].mean()
print("Cluster centers (standardized yeo-johnson-transformed features):")
display(cluster_means_scaled.round(3))

cluster_means_scaled.T.plot(kind="bar", figsize=(16, 8), edgecolor="white")
plt.title("Cluster Profiles: Mean Standardized Feature Values (Full Data)")
plt.ylabel("Standardized Value (yeo-johnson-transformed)")
plt.xlabel("Feature")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.heatmap(cluster_means_scaled, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, cbar_kws={"label": "Standardized Mean"})
plt.title("Cluster Center Heatmap (Full Data)")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()


In [ ]:
# Cluster profiles in original units
profile_orig = df_reviews.copy()
profile_orig["Cluster"] = cluster_labels

cluster_means_orig = profile_orig.groupby("Cluster")[reviews_cols].mean()
print("Cluster means in original units:")
display(cluster_means_orig.round(1))

cluster_means_orig.T.plot(kind="bar", figsize=(16, 8), edgecolor="white")
plt.title("Cluster Profiles: Centers in Original Units (Full Data)")
plt.ylabel("Original Units")
plt.xlabel("Feature")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.tight_layout()
plt.show()

In [ ]:
# PCA projection for visualization (full data)
pca_vis = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca_vis = pca_vis.fit_transform(X_scaled_full)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca_vis[:, 0], X_pca_vis[:, 1], c=cluster_labels,
                      cmap="Set2", alpha=0.7, edgecolors="k", linewidth=0.3)
plt.colorbar(scatter, label="Cluster")
plt.xlabel(f"PC1 ({pca_vis.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca_vis.explained_variance_ratio_[1]:.1%} variance)")
plt.title("K-Means Clusters Projected onto First Two Principal Components (Full Data)")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------
# Supervised Augmentation (on full data)
# Target = cluster_labels from KMeans on full data
# ---------------------------------------------------

# Multinomial Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
lr.fit(X_scaled_full, cluster_labels)

cv_scores_lr = cross_val_score(lr, X_scaled_full, cluster_labels, cv=5, scoring="accuracy")
print(f"Logistic Regression -- 5-fold CV Accuracy: {cv_scores_lr.mean():.3f} (+/- {cv_scores_lr.std():.3f})")

coef_df = pd.DataFrame(lr.coef_, columns=reviews_cols,
                       index=[f"Cluster {i}" for i in range(k_chosen)])

plt.figure(figsize=(14, 6))
sns.heatmap(coef_df, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, cbar_kws={"label": "Log-Odds Coefficient"})
plt.title("Multinomial Logistic Regression -- Coefficients by Cluster (Full Data)")
plt.ylabel("Cluster (vs. reference)")
plt.tight_layout()
plt.show()

In [ ]:
coef_df.T.plot(kind="bar", figsize=(16, 9), edgecolor="white")
plt.title("Multinomial Logistic Regression -- Coefficients by Cluster (Full Data)")
plt.ylabel("Log-Odds Coefficient")
plt.xlabel("Feature")
plt.axhline(0, color="black", linewidth=0.8, linestyle="--")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Decision tree
dt = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_SEED)
dt.fit(X_scaled_full, cluster_labels)

cv_scores_dt = cross_val_score(dt, X_scaled_full, cluster_labels, cv=5, scoring="accuracy")
print(f"Decision Tree -- 5-fold CV Accuracy: {cv_scores_dt.mean():.3f} (+/- {cv_scores_dt.std():.3f})")

fig, ax = plt.subplots(figsize=(16, 9))
plot_tree(dt, feature_names=reviews_cols,
          class_names=[f"C{i}" for i in range(k_chosen)],
          filled=True, rounded=True, fontsize=9, ax=ax,
          proportion=True)
plt.title("Decision Tree Explaining Cluster Membership (Full Data)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(dt.feature_importances_, index=reviews_cols).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind="barh", color="teal", edgecolor="white")
plt.xlabel("Feature Importance (Gini)")
plt.title("Decision Tree -- Feature Importances for Cluster Membership (Full Data)")
plt.tight_layout()
plt.show()


## Additional Analysis
---
---

In [ ]:
# Overall engagement per cluster, computed from the fitted clusters
# (previously this was a pasted literal table, which broke on re-run).
engagement = cluster_means_orig.mean(axis=1).sort_index()

plt.figure(figsize=(8, 5))
colors = plt.cm.tab10(range(len(engagement)))
bars = plt.bar(engagement.index, engagement.values,
               color=colors, alpha=0.8, edgecolor='black')

plt.xlabel('Cluster', fontsize=12, fontweight='bold')
plt.ylabel('Average Rating Across All Categories', fontsize=12, fontweight='bold')
plt.title('Overall Engagement Level by Cluster', fontsize=14, fontweight='bold')
plt.xticks(engagement.index)
plt.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, engagement.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
             f'{val:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
cluster_pop = pd.Series(cluster_labels).value_counts().sort_index()
cluster_share = (cluster_pop / len(cluster_labels) * 100).round(1)

summary = pd.DataFrame({
    'n_users': cluster_pop,
    'share_pct': cluster_share,
    'mean_rating': cluster_means_orig.mean(axis=1).round(2),
})
summary.index.name = 'Cluster'
display(summary)

## Step 7: Segment Definition
---
---

In [ ]:
# Top and bottom categories per cluster, to support naming the segments.
# Values are standardized, so +/- is relative to the average reviewer.
for cl in sorted(cluster_means_scaled.index):
    row = cluster_means_scaled.loc[cl].sort_values(ascending=False)
    n = int((pd.Series(cluster_labels) == cl).sum())
    print(f'\nCluster {cl}  (n={n}, {n / len(cluster_labels):.1%})')
    print('  strongest: ' + ', '.join(f'{k} {v:+.2f}' for k, v in row.head(4).items()))
    print('  weakest:   ' + ', '.join(f'{k} {v:+.2f}' for k, v in row.tail(4).items()))

### Segment profiles and recommended packages

> **TO COMPLETE AFTER RE-RUNNING.** The cell above prints the defining categories for
> each cluster. Use it to fill the table below with a plain-language name and the travel
> package each segment should be offered. This is the section that turns the analysis
> into a recommendation — do not leave it as a placeholder.

| Cluster | Size | Defining preferences | Segment name | Recommended package |
|---|---|---|---|---|
| 0 | | | | |
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |
| 7 | | | | |

### Method notes and limitations

- **Solution quality.** Mean silhouette is low in absolute terms, which is expected for
  dense behavioural rating data — the segments are regions of a continuum, not separated
  groups. The practical test is whether the profiles are distinct and actionable, not
  whether silhouette is high.
- **Stability.** Resampling ARI across 30 runs indicates how reliably the same structure
  recovers on 80% subsamples. Report the mean and range.
- **Cross-method agreement.** ARI between k-means and Ward's hierarchical clustering
  measures whether the structure is method-dependent.
- **Supervised check.** The logistic regression and decision tree are fit to predict the
  cluster labels themselves, so high accuracy confirms the segments are cleanly separable
  and identifies which categories define them. It is not out-of-sample validation of the
  segmentation, and should not be read as such.
- **Exclusions.** Users with more than 25% zero ratings were removed before clustering.
- **Choice of k.** k was chosen from the elbow and silhouette diagnostics together with
  interpretability of the resulting profiles, not from a single metric.